# Train and export the GridToEV models

This notebook trains the complete prediction bundle used by the FastAPI service. It keeps
time in the correct order: the oldest 70% of issue times are used for training, the following
15% select thresholds and blending, and the newest 15% are held back for final evaluation.

The saved bundle contains seven fitted pipelines:

1. dispatch-down event classifier;
2. dispatch-down MWh change regressor;
3. curtailment MWh regressor;
4. network-constraint MWh regressor; and
5. three dispatch-down quantile regressors for the 10th, 50th, and 90th percentiles.

The API loads `models/gridtoev_model_bundle.joblib` once when it starts. It does not execute
this notebook for each prediction.

## Why the MWh prediction uses a hybrid

Dispatch-down changes slowly from one half-hour to the next, so the latest observed value is
a strong baseline. The machine-learning regressor predicts the *change* from that observation.
A blend weight is selected only on the validation period. This preserves a reliable baseline
while allowing weather, load, price, SNSP, and grid-state features to adjust the estimate.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from gridtoev.inference import PredictionService
from gridtoev.training import (
    TrainingConfig,
    chronological_split,
    feature_columns,
    load_dataset,
    train_and_save,
)

DATASET_PATH = REPO_ROOT / "data" / "processed" / "gridtoev_model_ready.csv"
MODEL_PATH = REPO_ROOT / "models" / "gridtoev_model_bundle.joblib"
METRICS_PATH = REPO_ROOT / "models" / "training_metrics.json"
METADATA_PATH = REPO_ROOT / "models" / "model_metadata.json"

## 1. Inspect the modelling contract

Timestamp columns identify when the forecast is made and what future interval is predicted.
Target columns are excluded automatically; every remaining numeric column is a model feature.

In [2]:
data = load_dataset(DATASET_PATH)
columns = feature_columns(data)
train, validation, test = chronological_split(data)

split_summary = pd.DataFrame(
    [
        {
            "partition": name,
            "rows": len(frame),
            "issue_time_start": frame["issue_timestamp_utc"].min(),
            "issue_time_end": frame["issue_timestamp_utc"].max(),
            "event_rate": frame["dispatch_down_event"].mean(),
        }
        for name, frame in (
            ("train", train),
            ("validation", validation),
            ("test", test),
        )
    ]
)

print(f"Dataset shape: {data.shape}")
print(f"Model features: {len(columns)}")
display(split_summary)

Dataset shape: (2867, 131)
Model features: 117


,partition,rows,issue_time_start,issue_time_end,event_rate
0,train,2006,2026-01-02 00:00:00+00:00,2026-01-22 22:00:00+00:00,0.324526
1,validation,430,2026-01-22 22:30:00+00:00,2026-01-27 10:30:00+00:00,0.802326
2,test,431,2026-01-27 11:00:00+00:00,2026-01-31 22:30:00+00:00,0.684455


## 2. Train, evaluate, and persist all models

The evaluation models see only the training partition. Validation chooses the classification
threshold, MWh blend weight, and uncertainty adjustment. The untouched test period then
measures performance. Finally, production copies are fitted on train plus validation and
saved with the exact feature order and dependency versions.

In [3]:
config = TrainingConfig(
    learning_rate=0.05,
    max_iter=180,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42,
)

result = train_and_save(
    dataset_path=DATASET_PATH,
    artifact_path=MODEL_PATH,
    metrics_path=METRICS_PATH,
    metadata_path=METADATA_PATH,
    config=config,
)

print(f"Saved model bundle: {result.artifact_path.relative_to(REPO_ROOT)}")
print(f"Bundle size: {result.artifact_path.stat().st_size / 1024 / 1024:.2f} MB")
print("Models:")
for model_name in result.bundle["models"]:
    print(f"  - {model_name}")

Saved model bundle: models\gridtoev_model_bundle.joblib
Bundle size: 1.63 MB
Models:
  - event_classifier
  - dispatch_down_regressor
  - curtailment_regressor
  - constraint_regressor
  - dispatch_down_quantile_p10
  - dispatch_down_quantile_p50
  - dispatch_down_quantile_p90


## 3. Review held-out metrics

Classification uses average precision, Brier score, precision, recall, and F1. MWh models use
MAE, RMSE, and WAPE where the test period contains positive target energy. The persistence
baseline is shown beside the trained hybrid so the model is not credited for simple temporal
continuity. Quantile coverage checks how often the real value falls inside the P10-P90 band.

In [4]:
classification_table = pd.DataFrame(result.metrics["classification"]).T
regression_table = pd.DataFrame(
    {
        name: values
        for name, values in result.metrics["regression"].items()
        if isinstance(values, dict)
    }
).T
uncertainty_table = pd.DataFrame(
    list(result.metrics["uncertainty"].items()),
    columns=["metric", "value"],
)

display(classification_table)
display(regression_table)
display(uncertainty_table)
print(
    "Selected MWh ML weight:",
    result.metrics["regression"]["dispatch_down_ml_blend_weight"],
)

,average_precision,brier_score,precision,recall,f1,threshold,event_rate,roc_auc
validation,0.996109,0.147024,0.981481,0.921739,0.950673,0.1,0.802326,0.985405
test,0.999525,0.009213,0.993266,1.000000,0.996622,0.1,0.684455,0.999003
persistence_baseline_test,0.974686,0.020882,0.976667,0.993220,0.984874,0.5,0.684455,0.970875


,mae,rmse,wape,actual_total_mwh,nonzero_rate
dispatch_down_mwh,18.852121,29.211578,0.274496,29600.69,0.684455
dispatch_down_ml_only,25.263186,39.631163,0.367844,29600.69,0.684455
dispatch_down_persistence_baseline,16.821787,28.148681,0.244933,29600.69,0.684455
curtailment_mwh,1.161012,6.068326,NaN,0.00,0.000000
constraint_mwh,44.328827,70.109065,0.645449,29600.69,0.684455


,metric,value
0,p10_pinball_loss,6.867910
1,p50_pinball_loss,34.339548
2,p90_pinball_loss,13.476739
3,p10_p90_empirical_coverage,0.714617
4,conformal_interval_adjustment_mwh,13.986060


Selected MWh ML weight: 0.4


## 4. Load the artifact exactly as the API does

This is an end-to-end smoke test. It reloads the serialized bundle, finds the latest issue time
available for both horizons, and produces frontend-ready JSON responses.

In [5]:
service = PredictionService(MODEL_PATH, DATASET_PATH)
service.load()

latest_predictions = service.predict_latest(flexible_load_capacity_mw=100.0)
display(pd.DataFrame(latest_predictions))
print(json.dumps(latest_predictions[0], indent=2))

,model_version,issue_timestamp_utc,target_timestamp_utc,forecast_horizon_minutes,dispatch_down_probability,dispatch_down_event_prediction,classification_threshold,risk_level,predicted_dispatch_down_mwh,predicted_curtailment_mwh,predicted_constraint_mwh,prediction_interval_p10_mwh,prediction_interval_p50_mwh,prediction_interval_p90_mwh,flexible_load_capacity_mw,recoverable_surplus_mwh
0,1.0.0,2026-01-31T22:00:00+00:00,2026-01-31T22:30:00+00:00,30,0.000554,False,0.1,low,0.820363,0.222286,0.598077,0.0,0.0,28.744356,100.0,0.820363
1,1.0.0,2026-01-31T22:00:00+00:00,2026-01-31T23:00:00+00:00,60,0.000474,False,0.1,low,0.852260,0.230929,0.621331,0.0,0.0,28.744356,100.0,0.852260


{
  "model_version": "1.0.0",
  "issue_timestamp_utc": "2026-01-31T22:00:00+00:00",
  "target_timestamp_utc": "2026-01-31T22:30:00+00:00",
  "forecast_horizon_minutes": 30,
  "dispatch_down_probability": 0.0005535189355641568,
  "dispatch_down_event_prediction": false,
  "classification_threshold": 0.1,
  "risk_level": "low",
  "predicted_dispatch_down_mwh": 0.8203631446304058,
  "predicted_curtailment_mwh": 0.22228629207201464,
  "predicted_constraint_mwh": 0.5980768525583912,
  "prediction_interval_p10_mwh": 0.0,
  "prediction_interval_p50_mwh": 0.0,
  "prediction_interval_p90_mwh": 28.744356155062047,
  "flexible_load_capacity_mw": 100.0,
  "recoverable_surplus_mwh": 0.8203631446304058
}


## Next step

Start the API from the repository root:

```powershell
uvicorn gridtoev.api:app --app-dir src --host 0.0.0.0 --port 8000
```

Open `http://localhost:8000/docs` to try the endpoints. A frontend can call
`GET /predict/latest` for an immediate demo or `POST /predict/from-dataset` for a selected
historical issue time. `POST /predict/features` is the production-facing contract for a live
feature service.